# B3 — Bank Customer Churn: Neural Network

**Laboratory Practice III (410246), Group B**

Dataset reference: https://www.kaggle.com/competitions/churn-modelling/data

Run cells top-to-bottom. Set `CSV_PATH` to your downloaded CSV file to use real data; `None` runs the included synthetic demonstration. Results on synthetic data are not real-world model performance.

In [ ]:
# Set to a local CSV filename/path after downloading the Kaggle dataset.
# Keep None to run with built-in synthetic demonstration data.
CSV_PATH = None  # Example: 'Churn_Modelling.csv'

## Implementation and evaluation

In [ ]:
"""B3: Bank customer churn neural network with normalized inputs and evaluation."""

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

from sklearn.datasets import make_classification
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier

def main():
    if CSV_PATH:  # Load bank customer churn CSV here.
        df=pd.read_csv(CSV_PATH)
    else:
        x,y=make_classification(n_samples=500,n_features=8,n_informative=5,random_state=42)
        df=pd.DataFrame(x,columns=[f'feature_{i}' for i in range(8)]);df['Exited']=y
        print('Using synthetic demonstration data.')
    target=next((c for c in ['Exited','exited','Churn','churn'] if c in df),None)
    if target is None:raise ValueError('Expected Exited or Churn target column')
    y=pd.to_numeric(df[target],errors='coerce');valid=y.notna() & y.isin([0,1]);df=df.loc[valid];y=y.loc[valid].astype(int)
    if y.nunique()!=2 or y.value_counts().min()<2:raise ValueError('Need both churn classes')
    X=df.drop(columns=[target,'RowNumber','CustomerId','Surname'],errors='ignore')
    X=X.dropna(axis=1,how='all')
    num=X.select_dtypes(include='number').columns.tolist();cat=[c for c in X if c not in num]
    transforms=[]
    if num:transforms.append(('numeric',Pipeline([('fill',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num))
    if cat:transforms.append(('categorical',Pipeline([('fill',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat))
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
    # Tune architecture using only training data; hold test data untouched.
    from sklearn.model_selection import GridSearchCV
    pipe=Pipeline([('prep',ColumnTransformer(transforms)),('nn',MLPClassifier(max_iter=400,early_stopping=True,random_state=42))])
    search=GridSearchCV(pipe,{'nn__hidden_layer_sizes':[(32,),(64,32)],'nn__alpha':[.001,.01]},cv=3,scoring='accuracy',n_jobs=-1)
    search.fit(Xtr,ytr);pred=search.predict(Xte)
    print('Best hyperparameters:',search.best_params_)
    print('Accuracy:',accuracy_score(yte,pred));print('Confusion matrix:\n',confusion_matrix(yte,pred))
main()
